# Fixed prompt generation


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path('/kaggle/working/PA2')
assert (REPO_ROOT / 'generate_output.py').is_file(), (
    'Place your completed repository at /kaggle/working/PA2 before running this notebook.'
)
%cd /kaggle/working/PA2
sys.path.insert(0, str(REPO_ROOT))


In [ ]:
print('Using the already-installed, tested packages.')


In [ ]:
import gc
import json
import shutil
import torch

from data_utils import set_seed
from generate_output import create_alpaca_eval_subset, generate_dataset_outputs, load_model_and_tokenizer, save_generations
from persona_lora_training import PERSONA_EVAL_PROMPTS

set_seed(42)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator for generation.'
print('GPU:', torch.cuda.get_device_name(0))

EVAL_ROOT = REPO_ROOT / 'outputs' / 'eval'
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
TRAINING_OUTPUTS_OVERRIDE = Path('/kaggle/working/PA2/outputs')

def find_training_outputs():
    if TRAINING_OUTPUTS_OVERRIDE is not None:
        root = Path(TRAINING_OUTPUTS_OVERRIDE)
        for candidate in (root, root / 'outputs'):
            if (candidate / 'sft' / 'checkpoint-best').is_dir():
                return candidate
        raise FileNotFoundError(f'No sft/checkpoint-best found under {root}')

    input_root = Path('/kaggle/input')
    if input_root.exists():
        for checkpoint in input_root.rglob('checkpoint-best'):
            if checkpoint.parent.name == 'sft':
                candidate = checkpoint.parent.parent
                if (candidate / 'dpo_hh' / 'checkpoint-best').is_dir():
                    return candidate
        for archive_path in input_root.rglob('training_outputs.zip'):
            extract_root = Path('/kaggle/working/attached_training_outputs')
            extract_root.mkdir(parents=True, exist_ok=True)
            shutil.unpack_archive(str(archive_path), str(extract_root))
            for candidate in (extract_root, extract_root / 'outputs'):
                if (candidate / 'sft' / 'checkpoint-best').is_dir():
                    return candidate
    raise FileNotFoundError(
        'Training outputs were not found. Attach the training notebook output dataset, ' 
        'or set TRAINING_OUTPUTS_OVERRIDE to its outputs directory.'
    )

TRAINING_OUTPUTS = find_training_outputs()
print('Using training outputs:', TRAINING_OUTPUTS)
SFT_BEST = TRAINING_OUTPUTS / 'sft' / 'checkpoint-best'
HH_BEST = TRAINING_OUTPUTS / 'dpo_hh' / 'checkpoint-best'
UF_BEST = TRAINING_OUTPUTS / 'dpo_ultrafeedback' / 'checkpoint-best'
PERSONA_BEST = TRAINING_OUTPUTS / 'persona_lora'
for path in (SFT_BEST, HH_BEST, UF_BEST, PERSONA_BEST):
    assert path.exists(), f'Missing required checkpoint: {path}'

GREEDY_SETTINGS = {
    'max_new_tokens': 256, 'do_sample': False, 'temperature': 1.0, 'top_p': 1.0,
    'repetition_penalty': 1.0, 'no_repeat_ngram_size': 0,
}
PERSONA_SETTINGS = {**GREEDY_SETTINGS, 'max_new_tokens': 160}


In [ ]:
subset_path = EVAL_ROOT / 'alpaca_eval_300_seed42.json'
if subset_path.is_file():
    alpaca_rows = json.loads(subset_path.read_text(encoding='utf-8'))
    if len(alpaca_rows) != 300:
        raise ValueError(f'Expected 300 saved AlpacaEval prompts, found {len(alpaca_rows)}.')
else:
    alpaca_rows = create_alpaca_eval_subset(n=300, seed=42)
    subset_path.write_text(json.dumps(alpaca_rows, ensure_ascii=False, indent=2), encoding='utf-8')
print('AlpacaEval prompts:', len(alpaca_rows))

# Quick preflight on two prompts before running all 300.
for checkpoint, label in ((SFT_BEST, 'SFT'), (UF_BEST, 'UltraFeedback DPO')):
    model, tokenizer = load_model_and_tokenizer(str(checkpoint))
    sample = generate_dataset_outputs(
        model, tokenizer, alpaca_rows[:2], generator=label, dataset_name='alpaca_eval_sanity',
        **GREEDY_SETTINGS,
    )
    print('\nMODEL:', label)
    for row in sample:
        print('PROMPT:', row['instruction'])
        print('OUTPUT:', row['output'][:1000], '\n')
    del model, tokenizer, sample
    gc.collect()
    torch.cuda.empty_cache()


## Generate and save fixed evaluation outputs

Existing files are reused only when their IDs match the current prompt set. Each model's JSON is saved as soon as its generation run finishes.


In [ ]:
def run_generation(checkpoint, rows, generator, dataset_name, filename, settings):
    output_path = EVAL_ROOT / filename
    expected_ids = [str(row.get('id', f'{dataset_name}-{i:04d}')) for i, row in enumerate(rows)]
    if output_path.is_file():
        existing = json.loads(output_path.read_text(encoding='utf-8'))
        existing_ids = [str(row.get('id')) for row in existing]
        if existing_ids == expected_ids:
            print('Reusing', filename, '(', len(existing), 'rows )')
            return existing
        print('Existing file does not match prompt IDs; regenerating:', filename)

    model, tokenizer = load_model_and_tokenizer(str(checkpoint))
    try:
        outputs = generate_dataset_outputs(
            model, tokenizer, rows, generator=generator, dataset_name=dataset_name, **settings
        )
        save_generations(outputs, output_path)
    finally:
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
    print('Saved', output_path, 'rows=', len(outputs))
    return outputs


BASE_MODEL = 'Qwen/Qwen2.5-0.5B'
run_generation(BASE_MODEL, alpaca_rows, 'qwen2.5-0.5b-base', 'alpaca_eval', 'alpaca_base.json', GREEDY_SETTINGS)
run_generation(SFT_BEST, alpaca_rows, 'SFT', 'alpaca_eval', 'alpaca_sft.json', GREEDY_SETTINGS)
run_generation(UF_BEST, alpaca_rows, 'UltraFeedback DPO', 'alpaca_eval', 'alpaca_dpo_ultrafeedback.json', GREEDY_SETTINGS)


hh_eval_path = REPO_ROOT / 'eval_sets' / 'hh_rlhf_eval_300_seed42.json'
assert hh_eval_path.is_file(), f'Missing supplied HH evaluation set: {hh_eval_path}'
hh_rows = json.loads(hh_eval_path.read_text(encoding='utf-8'))
if not isinstance(hh_rows, list) or not hh_rows:
    raise ValueError('The HH evaluation file must contain a non-empty JSON list.')
print('HH evaluation prompts:', len(hh_rows))
run_generation(SFT_BEST, hh_rows, 'SFT', 'hh_rlhf_style_eval', 'hh_sft.json', GREEDY_SETTINGS)
run_generation(HH_BEST, hh_rows, 'HH-RLHF DPO', 'hh_rlhf_style_eval', 'hh_dpo.json', GREEDY_SETTINGS)

run_generation('HuggingFaceTB/SmolLM2-360M-Instruct', PERSONA_EVAL_PROMPTS, 'SmolLM2-360M-Instruct base', 'persona_style_eval', 'persona_base_eval.json', PERSONA_SETTINGS)
run_generation(PERSONA_BEST, PERSONA_EVAL_PROMPTS, 'Persona LoRA DPO', 'persona_style_eval', 'persona_lora_dpo_eval.json', PERSONA_SETTINGS)

manifest = {
    'seed': 42, 'alpaca_eval_count': len(alpaca_rows), 'hh_eval_count': len(hh_rows),
    'greedy_decoding': True, 'alpaca_settings': GREEDY_SETTINGS, 'persona_settings': PERSONA_SETTINGS,
    'training_outputs': str(TRAINING_OUTPUTS),
}
(EVAL_ROOT / 'generation_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Skipping generation ZIP to preserve disk space.')
print('Generation files:', EVAL_ROOT)

